# rgcn / lsep / total / n08

```
run      rgcn-lsep-total-n08
purpose  normalization / total degree
bucket   exapt-lab/hangar/runs/rgcn-lsep-total-n08
```

In [1]:
REPOSITORY = "https://github.com/aekn/exapt.git"
GIT_SHA = "cda583e5a90386f67f03e1208d14ab00f7405d6d"
DATA_SHA = "a29db6bb7019eefa4fd93d7b771f212cc88014a1"

RUN = "rgcn-lsep-total-n08"
PURPOSE = "normalization / total degree"

OBJECTIVE = "lsep"
NORMALIZATION = "total"
NEGATIVES = 8

WIDTH = 128
SEED = 101
EPOCHS = 120

LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
DROPOUT = 0.0

In [2]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import userdata


ROOT = Path("/content/exapt")
OUTPUT = Path("/content/runs") / RUN
PYTHON = ROOT / ".venv/bin/python"
HF = ROOT / ".venv/bin/hf"

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)

if not (ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", "--quiet", REPOSITORY, str(ROOT)], check=True)

subprocess.run(["git", "-C", str(ROOT), "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", "--quiet", "--detach", GIT_SHA], check=True)
subprocess.run(["uv", "python", "install", "3.12", "--quiet"], check=True)
subprocess.run(["uv", "sync", "--locked", "--extra", "train", "--no-dev", "--python", "3.12", "--quiet"], cwd=ROOT, check=True)

CompletedProcess(args=['uv', 'sync', '--locked', '--extra', 'train', '--no-dev', '--python', '3.12', '--quiet'], returncode=0)

In [3]:
def train(epochs: int) -> None:
    token = userdata.get("exapt_hf")
    if not token:
        raise RuntimeError("missing Colab secret: exapt_hf")

    env = {
        **os.environ,
        "HF_TOKEN": token,
        "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    }

    print("source   fetch primekg", flush=True)

    snapshot = subprocess.check_output(
        [
            str(HF), "download", "exapt-lab/data",
            "sources/primekg-2.1/manifest.json",
            "sources/primekg-2.1/kg.csv",
            "--repo-type", "dataset",
            "--revision", DATA_SHA,
            "--quiet",
        ],
        cwd=ROOT,
        env=env,
        text=True,
    ).strip()
    source = Path(snapshot) / "sources/primekg-2.1"
    print("source   ready", flush=True)

    command = [
        str(PYTHON), "-u", "scripts/fit.py", str(source), str(OUTPUT),
        "--encoder", "rgcn",
        "--width", str(WIDTH),
        "--normalization", NORMALIZATION,
        "--layout", "csr",
        "--features", "trainable",
        "--dropout", str(DROPOUT),
        "--objective", OBJECTIVE,
        "--candidate-pool", "sampled",
        "--seed", str(SEED),
        "--epochs", str(epochs),
        "--negatives", str(NEGATIVES),
        "--negative-weight", "1",
        "--learning-rate", str(LEARNING_RATE),
        "--weight-decay", str(WEIGHT_DECAY),
        "--scheduler", "constant",
        "--benchmark-seed", "163",
        "--folds", "4",
        "--development-fraction", "0.1",
        "--test-fraction", "0.1",
        "--k", "20",
        "--device", "cuda",
        "--dataset-revision", DATA_SHA,
        "--purpose", PURPOSE,
        "--bucket-run", RUN,
    ]
    if (OUTPUT / "resume.pt").is_file():
        command.append("--resume")

    print("train    launch", flush=True)

    with subprocess.Popen(
        command,
        cwd=ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        if process.stdout is None:
            raise RuntimeError("training output pipe unavailable")
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()

    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

In [4]:
train(EPOCHS)

source   fetch primekg
source   ready
train    launch
exapt / rgcn-lsep-total-n08

model    rgcn / distmult
train    lsep / total / n8 / w128
seed     101
folds    4
budget   120 epochs
device   cuda / NVIDIA L4
local    /content/runs/rgcn-lsep-total-n08
bucket   exapt-lab/hangar/runs/rgcn-lsep-total-n08

source   load therapeutics
source   ready / 17,080 diseases
graph    load source arcs
graph    ready / 8,100,498 arcs
folds    prepare masked contexts

epoch       loss         lr      R@20   nDCG@20       wall
 0001     3.3283   1.00e-03    0.0024    0.0022   00:00:07   *
 0002     3.1194   1.00e-03    0.0155    0.0094   00:00:06   *
 0003     2.8548   1.00e-03    0.0293    0.0239   00:00:06   *
 0004     2.6119   1.00e-03    0.0656    0.0370   00:00:06   *
 0005     2.3662   1.00e-03    0.0504    0.0234   00:00:06
 0006     2.0874   1.00e-03    0.0696    0.0323   00:00:06   *
 0007     1.8223   1.00e-03    0.1163    0.0530   00:00:06   *
 0008     1.5456   1.00e-03    0.1110    0.07